In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/aisehack-2-0/sample_submission.csv
/kaggle/input/competitions/aisehack-2-0/base_line_model.ipynb
/kaggle/input/competitions/aisehack-2-0/train.csv
/kaggle/input/competitions/aisehack-2-0/test.csv


## Setup

Imports and reproducibility settings.


In [2]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.base import clone
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, RobustScaler, StandardScaler
from sklearn.svm import SVR

warnings.filterwarnings("ignore")

SEED = 42
N_SPLITS = 5
TRY_INSTALL_RDKIT = os.environ.get("TRY_INSTALL_RDKIT", "1") == "1"

In [3]:
def find_input_file(filename):
    input_dir = os.environ.get("POLYMER_INPUT_DIR")
    if input_dir:
        candidate = Path(input_dir) / filename
        if candidate.exists():
            return candidate

    local = Path(filename)
    if local.exists():
        return local

    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        matches = list(kaggle_root.rglob(filename))
        if matches:
            return matches[0]

    raise FileNotFoundError(f"Could not find {filename}.")

In [4]:
try:
    from rdkit import Chem, DataStructs, RDLogger
    from rdkit.Chem import AllChem, Crippen, Descriptors, Lipinski, MACCSkeys, rdMolDescriptors
    RDLogger.DisableLog("rdApp.*")
    RDKIT_AVAILABLE = True
    print("RDKit available")
except Exception as exc:
    if TRY_INSTALL_RDKIT:
        try:
            import subprocess
            import sys
            print("RDKit missing; installing public rdkit package inside notebook...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdkit"])
            from rdkit import Chem, DataStructs, RDLogger
            from rdkit.Chem import AllChem, Crippen, Descriptors, Lipinski, MACCSkeys, rdMolDescriptors
            RDLogger.DisableLog("rdApp.*")
            RDKIT_AVAILABLE = True
            print("RDKit installed and imported")
        except Exception as install_exc:
            RDKIT_AVAILABLE = False
            print(f"RDKit unavailable: {exc}")
            print(f"RDKit install failed: {install_exc}")
    else:
        RDKIT_AVAILABLE = False
        print(f"RDKit unavailable: {exc}")

DESCRIPTOR_FUNCS = list(Descriptors._descList) if RDKIT_AVAILABLE else []

RDKit missing; installing public rdkit package inside notebook...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 46.2 MB/s eta 0:00:00
RDKit installed and imported


## SMILES preprocessing

Input normalization and string-level molecular views.


In [5]:
def to_dense(X):
    return X.toarray() if sparse.issparse(X) else X

In [6]:
def safe_mol(smiles):
    if not RDKIT_AVAILABLE or pd.isna(smiles):
        return None
    try:
        return Chem.MolFromSmiles(str(smiles))
    except Exception:
        return None

In [7]:
def smiles_views(smiles_series):
    smiles = smiles_series.fillna("").astype(str)
    nostar = smiles.str.replace("*", "", regex=False)
    alpha = nostar.str.replace(r"[=#@/\\\-\+\[\]\(\)0-9]", "", regex=True)
    left = smiles.str.split("*", n=1).str[0]
    right = smiles.str.rsplit("*", n=1).str[-1]
    middle = smiles.str.replace(r"^\*", "", regex=True).str.replace(r"\*$", "", regex=True)
    canonical = []
    canonical_nostar = []
    for s in smiles:
        mol = safe_mol(s)
        mol_nostar = safe_mol(str(s).replace("*", ""))
        try:
            canonical.append(Chem.MolToSmiles(mol, canonical=True) if mol is not None else str(s))
        except Exception:
            canonical.append(str(s))
        try:
            canonical_nostar.append(Chem.MolToSmiles(mol_nostar, canonical=True) if mol_nostar is not None else str(s).replace("*", ""))
        except Exception:
            canonical_nostar.append(str(s).replace("*", ""))
    canonical = pd.Series(canonical, index=smiles.index)
    canonical_nostar = pd.Series(canonical_nostar, index=smiles.index)
    return {
        "raw": smiles,
        "nostar": nostar,
        "alpha": alpha,
        "left": left,
        "right": right,
        "middle": middle,
        "canonical": canonical,
        "canonical_nostar": canonical_nostar,
    }

In [8]:
def smiles_string_features(smiles_series):
    smiles = smiles_series.fillna("").astype(str)
    feats = pd.DataFrame(index=smiles.index)
    feats["smiles_len"] = smiles.str.len()
    feats["log_smiles_len"] = np.log1p(feats["smiles_len"])
    feats["sqrt_smiles_len"] = np.sqrt(feats["smiles_len"])
    feats["num_atoms_tokens"] = smiles.str.count(r"[A-Z][a-z]?")
    feats["num_branches"] = smiles.str.count(r"\(") + smiles.str.count(r"\)")
    feats["num_rings"] = smiles.str.count(r"[0-9]")
    feats["num_double"] = smiles.str.count("=")
    feats["num_triple"] = smiles.str.count("#")
    feats["num_aromatic_c"] = smiles.str.count("c")
    feats["num_aromatic_n"] = smiles.str.count("n")
    feats["num_star"] = smiles.str.count(r"\*")
    feats["num_brackets"] = smiles.str.count(r"\[") + smiles.str.count(r"\]")

    atom_weights = {"C": 12.011, "N": 14.007, "O": 15.999, "S": 32.06, "F": 18.998,
                    "Cl": 35.45, "Br": 79.904, "I": 126.904, "P": 30.974, "Si": 28.085, "B": 10.81}
    atom_pattern = r"Cl|Br|Si|[BCNOFPSIbcnops]"
    rows = []
    for s in smiles:
        counts = {atom: 0 for atom in atom_weights}
        aromatic_atoms = 0
        for atom in pd.Series([s]).str.findall(atom_pattern).iloc[0]:
            normalized = atom.capitalize() if len(atom) == 1 else atom
            if normalized in counts:
                counts[normalized] += 1
            if atom.islower():
                aromatic_atoms += 1
        counts["aromatic_atoms"] = aromatic_atoms
        rows.append(counts)
    atom_df = pd.DataFrame(rows, index=smiles.index).fillna(0.0)
    for atom in atom_weights:
        feats[f"atom_{atom}"] = atom_df[atom]
    feats["approx_mol_weight"] = sum(atom_df[atom] * weight for atom, weight in atom_weights.items())
    feats["approx_heavy_atoms"] = atom_df[list(atom_weights)].sum(axis=1)
    feats["aromatic_atom_count"] = atom_df["aromatic_atoms"]
    feats["hetero_atom_count"] = atom_df[["N", "O", "S", "F", "Cl", "Br", "I", "P", "Si", "B"]].sum(axis=1)
    feats["halogen_count"] = atom_df[["F", "Cl", "Br", "I"]].sum(axis=1)
    feats["hetero_ratio"] = feats["hetero_atom_count"] / feats["approx_heavy_atoms"].replace(0, 1)
    feats["aromatic_ratio"] = feats["aromatic_atom_count"] / feats["approx_heavy_atoms"].replace(0, 1)

    functional_groups = {
        "fg_ester": r"C\(=O\)O|OC\(=O\)",
        "fg_amide": r"C\(=O\)N|NC\(=O\)",
        "fg_imide": r"N\(C\(=O\).+C\(=O\)\)|N1C\(=O\).+C1=O",
        "fg_sulfone": r"S\(=O\)\(=O\)",
        "fg_ether": r"COC|cOc|COc|cOC",
        "fg_nitrile": r"C#N",
        "fg_nitro": r"\[N\+\]\(=O\)\[O-\]",
        "fg_siloxane": r"Si.*O.*Si|O\[Si\]",
        "fg_phenyl": r"c1ccccc1|c1ccc",
        "fg_carbonyl": r"C\(=O\)",
    }
    for name, pattern in functional_groups.items():
        feats[name] = smiles.str.count(pattern)

    for token in ["C", "N", "O", "S", "F", "Cl", "Br", "I", "P", "Si"]:
        feats[f"token_{token}"] = smiles.str.count(token)

    denom = feats["smiles_len"].replace(0, 1)
    for col in list(feats.columns):
        if col.startswith("num_") or col.startswith("token_") or col.startswith("fg_") or col.startswith("atom_"):
            feats[f"{col}_ratio"] = feats[col] / denom
    return feats.astype(np.float32)

## Polymer descriptors

Backbone, conjugation, and standard RDKit descriptors.


In [9]:
def polymer_electronic_features(mol):
    atoms = list(mol.GetAtoms()); bonds = list(mol.GetBonds())
    n_atoms = max(len(atoms), 1); n_bonds = max(len(bonds), 1)
    conjugated = [b for b in bonds if b.GetIsConjugated() or b.GetIsAromatic()]
    aromatic_atoms = [a for a in atoms if a.GetIsAromatic()]
    aromatic_hetero = [a for a in aromatic_atoms if a.GetAtomicNum() not in (0, 6)]

    adjacency = {a.GetIdx(): [] for a in atoms}
    for bond in conjugated:
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        adjacency[i].append(j); adjacency[j].append(i)
    seen=set(); component_sizes=[]; max_diameter=0
    for start in adjacency:
        if start in seen or not adjacency[start]: continue
        stack=[start]; seen.add(start); component=[]
        while stack:
            node=stack.pop(); component.append(node)
            for nxt in adjacency[node]:
                if nxt not in seen: seen.add(nxt); stack.append(nxt)
        component_sizes.append(len(component))
        comp_set=set(component)
        for source in component:
            dist={source:0}; queue=[source]
            for node in queue:
                for nxt in adjacency[node]:
                    if nxt in comp_set and nxt not in dist:
                        dist[nxt]=dist[node]+1; queue.append(nxt)
            max_diameter=max(max_diameter,max(dist.values(),default=0))

    result = {
        "poly_conjugated_bonds": len(conjugated),
        "poly_conjugated_bond_fraction": len(conjugated)/n_bonds,
        "poly_aromatic_atom_fraction": len(aromatic_atoms)/n_atoms,
        "poly_aromatic_hetero_count": len(aromatic_hetero),
        "poly_largest_conjugated_component": max(component_sizes,default=0),
        "poly_conjugated_diameter": max_diameter,
        "poly_double_bond_fraction": sum(b.GetBondType()==Chem.BondType.DOUBLE for b in bonds)/n_bonds,
        "poly_triple_bond_fraction": sum(b.GetBondType()==Chem.BondType.TRIPLE for b in bonds)/n_bonds,
        "poly_ring_bond_fraction": sum(b.IsInRing() for b in bonds)/n_bonds,
    }

    dummy = [a.GetIdx() for a in atoms if a.GetAtomicNum()==0]
    if len(dummy)>=2:
        path=list(Chem.GetShortestPath(mol,dummy[0],dummy[1])); path_set=set(path)
        path_bonds=[]
        for left,right in zip(path[:-1],path[1:]):
            bond=mol.GetBondBetweenAtoms(left,right)
            if bond is not None: path_bonds.append(bond)
        plen=max(len(path_bonds),1)
        internal=[mol.GetAtomWithIdx(i) for i in path if mol.GetAtomWithIdx(i).GetAtomicNum()!=0]
        result.update({
            "poly_backbone_length": len(path_bonds),
            "poly_backbone_conjugated_fraction": sum(b.GetIsConjugated() or b.GetIsAromatic() for b in path_bonds)/plen,
            "poly_backbone_aromatic_bond_fraction": sum(b.GetIsAromatic() for b in path_bonds)/plen,
            "poly_backbone_double_fraction": sum(b.GetBondType()==Chem.BondType.DOUBLE for b in path_bonds)/plen,
            "poly_backbone_ring_atom_fraction": sum(a.IsInRing() for a in internal)/max(len(internal),1),
            "poly_backbone_hetero_fraction": sum(a.GetAtomicNum() not in (0,6) for a in internal)/max(len(internal),1),
            "poly_off_backbone_fraction": sum(a.GetIdx() not in path_set for a in atoms)/n_atoms,
        })
    else:
        for name in ["poly_backbone_length","poly_backbone_conjugated_fraction",
                     "poly_backbone_aromatic_bond_fraction","poly_backbone_double_fraction",
                     "poly_backbone_ring_atom_fraction","poly_backbone_hetero_fraction",
                     "poly_off_backbone_fraction"]:
            result[name]=0.0
    return result

In [10]:
def rdkit_descriptor_frame(smiles_series):
    rows = []
    for smiles in smiles_series:
        mol = safe_mol(smiles)
        if mol is None:
            values = {name: np.nan for name, _ in DESCRIPTOR_FUNCS}
            values.update({
                "rd_TPSA": np.nan, "rd_FractionCSP3": np.nan, "rd_NumRotatableBonds": np.nan,
                "rd_NumHAcceptors": np.nan, "rd_NumHDonors": np.nan, "rd_MolLogP": np.nan,
                "rd_RingCount": np.nan, "rd_AromaticRingCount": np.nan, "rd_AliphaticRingCount": np.nan,
                "rd_HeavyAtomCount": np.nan, "rd_MolWt": np.nan, "rd_ExactMolWt": np.nan,
            })
            rows.append(values)
            continue

        values = {}
        for name, func in DESCRIPTOR_FUNCS:
            try:
                values[name] = func(mol)
            except Exception:
                values[name] = np.nan
        try:
            values["rd_TPSA"] = rdMolDescriptors.CalcTPSA(mol)
            values["rd_FractionCSP3"] = Lipinski.FractionCSP3(mol)
            values["rd_NumRotatableBonds"] = Lipinski.NumRotatableBonds(mol)
            values["rd_NumHAcceptors"] = Lipinski.NumHAcceptors(mol)
            values["rd_NumHDonors"] = Lipinski.NumHDonors(mol)
            values["rd_MolLogP"] = Crippen.MolLogP(mol)
            values["rd_RingCount"] = rdMolDescriptors.CalcNumRings(mol)
            values["rd_AromaticRingCount"] = rdMolDescriptors.CalcNumAromaticRings(mol)
            values["rd_AliphaticRingCount"] = rdMolDescriptors.CalcNumAliphaticRings(mol)
            values["rd_HeavyAtomCount"] = mol.GetNumHeavyAtoms()
            values["rd_MolWt"] = Descriptors.MolWt(mol)
            values["rd_ExactMolWt"] = Descriptors.ExactMolWt(mol)
            values.update(polymer_electronic_features(mol))
        except Exception:
            pass
        rows.append(values)
    return pd.DataFrame(rows, index=smiles_series.index).astype(np.float32)

## Fingerprints

Reusable binary and count fingerprint generators.


In [11]:
def morgan_bits(smiles_series, radius=2, n_bits=2048):
    if not RDKIT_AVAILABLE:
        return sparse.csr_matrix((len(smiles_series), 0), dtype=np.float32)
    arr = np.zeros((len(smiles_series), n_bits), dtype=np.uint8)
    for i, smiles in enumerate(smiles_series):
        mol = safe_mol(smiles)
        if mol is None:
            continue
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
        DataStructs.ConvertToNumpyArray(fp, arr[i])
    return sparse.csr_matrix(arr, dtype=np.float32)

In [12]:
def maccs_bits(smiles_series):
    if not RDKIT_AVAILABLE:
        return sparse.csr_matrix((len(smiles_series), 0), dtype=np.float32)
    arr = np.zeros((len(smiles_series), 167), dtype=np.uint8)
    for i, smiles in enumerate(smiles_series):
        mol = safe_mol(smiles)
        if mol is None:
            continue
        fp = MACCSkeys.GenMACCSKeys(mol)
        DataStructs.ConvertToNumpyArray(fp, arr[i])
    return sparse.csr_matrix(arr, dtype=np.float32)

In [13]:
def rdkit_path_bits(smiles_series, n_bits=2048):
    if not RDKIT_AVAILABLE:
        return sparse.csr_matrix((len(smiles_series), 0), dtype=np.float32)
    arr = np.zeros((len(smiles_series), n_bits), dtype=np.uint8)
    for i, smiles in enumerate(smiles_series):
        mol = safe_mol(smiles)
        if mol is None:
            continue
        fp = Chem.RDKFingerprint(mol, fpSize=n_bits)
        DataStructs.ConvertToNumpyArray(fp, arr[i])
    return sparse.csr_matrix(arr, dtype=np.float32)

In [14]:
def atom_pair_bits(smiles_series, n_bits=2048):
    if not RDKIT_AVAILABLE:
        return sparse.csr_matrix((len(smiles_series), 0), dtype=np.float32)
    arr = np.zeros((len(smiles_series), n_bits), dtype=np.uint8)
    for i, smiles in enumerate(smiles_series):
        mol = safe_mol(smiles)
        if mol is None:
            continue
        fp = rdMolDescriptors.GetHashedAtomPairFingerprintAsBitVect(mol, nBits=n_bits)
        DataStructs.ConvertToNumpyArray(fp, arr[i])
    return sparse.csr_matrix(arr, dtype=np.float32)

In [15]:
def torsion_bits(smiles_series, n_bits=2048):
    if not RDKIT_AVAILABLE:
        return sparse.csr_matrix((len(smiles_series), 0), dtype=np.float32)
    arr = np.zeros((len(smiles_series), n_bits), dtype=np.uint8)
    for i, smiles in enumerate(smiles_series):
        mol = safe_mol(smiles)
        if mol is None:
            continue
        fp = rdMolDescriptors.GetHashedTopologicalTorsionFingerprintAsBitVect(mol, nBits=n_bits)
        DataStructs.ConvertToNumpyArray(fp, arr[i])
    return sparse.csr_matrix(arr, dtype=np.float32)

In [16]:
def morgan_counts(smiles_series, radius=2, n_bits=2048):
    """Hashed fragment counts retain repeat frequency lost by binary fingerprints."""
    if not RDKIT_AVAILABLE:
        return sparse.csr_matrix((len(smiles_series), 0), dtype=np.float32)
    arr = np.zeros((len(smiles_series), n_bits), dtype=np.float32)
    for i, smiles in enumerate(smiles_series):
        mol = safe_mol(smiles)
        if mol is None:
            continue
        fp = AllChem.GetHashedMorganFingerprint(mol, radius, nBits=n_bits)
        for bit, count in fp.GetNonzeroElements().items():
            arr[i, bit] = np.log1p(count)
    return sparse.csr_matrix(arr)

## Feature matrices

Train/test feature construction.


In [17]:
def dense_descriptor_pair(train_df, test_df):
    train_dense = smiles_string_features(train_df["smiles"])
    test_dense = smiles_string_features(test_df["smiles"])
    if RDKIT_AVAILABLE:
        train_dense = pd.concat([train_dense, rdkit_descriptor_frame(train_df["smiles"])], axis=1)
        test_dense = pd.concat([test_dense, rdkit_descriptor_frame(test_df["smiles"])], axis=1)

    train_dense = train_dense.replace([np.inf, -np.inf], np.nan)
    test_dense = test_dense.replace([np.inf, -np.inf], np.nan)
    med = train_dense.median(numeric_only=True).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    train_dense = train_dense.fillna(med).fillna(0.0)
    test_dense = test_dense.fillna(med).fillna(0.0)

    scaler = RobustScaler()
    X_train = scaler.fit_transform(train_dense).astype(np.float32)
    X_test = scaler.transform(test_dense).astype(np.float32)
    return X_train, X_test

In [18]:
def sparse_text_pair(train_df, test_df):
    train_views = smiles_views(train_df["smiles"])
    test_views = smiles_views(test_df["smiles"])
    train_text = (
        train_views["raw"] + " " + train_views["nostar"] + " " + train_views["left"] + " "
        + train_views["right"] + " " + train_views["middle"] + " "
        + train_views["canonical"] + " " + train_views["canonical_nostar"]
    )
    test_text = (
        test_views["raw"] + " " + test_views["nostar"] + " " + test_views["left"] + " "
        + test_views["right"] + " " + test_views["middle"] + " "
        + test_views["canonical"] + " " + test_views["canonical_nostar"]
    )

    char_vec = TfidfVectorizer(analyzer="char", ngram_range=(2, 8), min_df=1, max_features=140000,
                               sublinear_tf=True, lowercase=False, dtype=np.float32)
    all_text = pd.concat([train_text, test_text], axis=0).fillna("").astype(str)
    char_vec.fit(all_text)
    X_train_char = char_vec.transform(train_text.fillna(""))
    X_test_char = char_vec.transform(test_text.fillna(""))

    alpha_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 7), min_df=1, max_features=70000,
                                sublinear_tf=True, lowercase=False, dtype=np.float32)
    all_alpha = pd.concat([train_views["alpha"], test_views["alpha"]], axis=0).fillna("").astype(str)
    alpha_vec.fit(all_alpha)
    X_train_alpha = alpha_vec.transform(train_views["alpha"].fillna(""))
    X_test_alpha = alpha_vec.transform(test_views["alpha"].fillna(""))

    dense_train, dense_test = dense_descriptor_pair(train_df, test_df)
    scaler = StandardScaler().fit(dense_train)
    X_train_dense = sparse.csr_matrix(scaler.transform(dense_train).astype(np.float32))
    X_test_dense = sparse.csr_matrix(scaler.transform(dense_test).astype(np.float32))
    train_nostar = train_df["smiles"].fillna("").astype(str).str.replace("*", "", regex=False)
    test_nostar = test_df["smiles"].fillna("").astype(str).str.replace("*", "", regex=False)

    fp_train = sparse.hstack(
        [
            morgan_bits(train_df["smiles"], 1, 1024),
            morgan_bits(train_df["smiles"], 2, 2048),
            morgan_bits(train_df["smiles"], 3, 2048),
            morgan_bits(train_nostar, 2, 2048),
            rdkit_path_bits(train_df["smiles"], 2048),
            rdkit_path_bits(train_nostar, 2048),
            atom_pair_bits(train_df["smiles"], 2048),
            torsion_bits(train_df["smiles"], 2048),
            maccs_bits(train_df["smiles"]),
        ],
        format="csr",
    )
    fp_test = sparse.hstack(
        [
            morgan_bits(test_df["smiles"], 1, 1024),
            morgan_bits(test_df["smiles"], 2, 2048),
            morgan_bits(test_df["smiles"], 3, 2048),
            morgan_bits(test_nostar, 2, 2048),
            rdkit_path_bits(test_df["smiles"], 2048),
            rdkit_path_bits(test_nostar, 2048),
            atom_pair_bits(test_df["smiles"], 2048),
            torsion_bits(test_df["smiles"], 2048),
            maccs_bits(test_df["smiles"]),
        ],
        format="csr",
    )

    X_train = sparse.hstack([X_train_dense, X_train_char, X_train_alpha, fp_train], format="csr")
    X_test = sparse.hstack([X_test_dense, X_test_char, X_test_alpha, fp_test], format="csr")
    return X_train, X_test

In [19]:
def dense_rdkit_pair(train_df, test_df, include_counts=True):
    desc_train, desc_test = dense_descriptor_pair(train_df, test_df)
    train_nostar = train_df["smiles"].fillna("").astype(str).str.replace("*", "", regex=False)
    test_nostar = test_df["smiles"].fillna("").astype(str).str.replace("*", "", regex=False)
    train_blocks = [
            morgan_bits(train_df["smiles"], 1, 512),
            morgan_bits(train_df["smiles"], 2, 1024),
            morgan_bits(train_df["smiles"], 3, 1024),
            morgan_bits(train_nostar, 2, 1024),
            rdkit_path_bits(train_df["smiles"], 1024),
            rdkit_path_bits(train_nostar, 1024),
            atom_pair_bits(train_df["smiles"], 1024),
            torsion_bits(train_df["smiles"], 1024),
            maccs_bits(train_df["smiles"]),
    ]
    test_blocks = [
            morgan_bits(test_df["smiles"], 1, 512),
            morgan_bits(test_df["smiles"], 2, 1024),
            morgan_bits(test_df["smiles"], 3, 1024),
            morgan_bits(test_nostar, 2, 1024),
            rdkit_path_bits(test_df["smiles"], 1024),
            rdkit_path_bits(test_nostar, 1024),
            atom_pair_bits(test_df["smiles"], 1024),
            torsion_bits(test_df["smiles"], 1024),
            maccs_bits(test_df["smiles"]),
    ]
    if include_counts:
        train_blocks += [morgan_counts(train_df["smiles"], 2, 1024), morgan_counts(train_nostar, 2, 1024)]
        test_blocks += [morgan_counts(test_df["smiles"], 2, 1024), morgan_counts(test_nostar, 2, 1024)]
    fp_train = sparse.hstack(
        train_blocks,
        format="csr",
    ).toarray().astype(np.float32)
    fp_test = sparse.hstack(
        test_blocks,
        format="csr",
    ).toarray().astype(np.float32)
    return np.hstack([desc_train, fp_train]).astype(np.float32), np.hstack([desc_test, fp_test]).astype(np.float32)

In [20]:
def build_feature_bundle(train_df, test_df, target_type):
    X_dense, X_dense_test = dense_rdkit_pair(train_df, test_df, include_counts=(target_type == "tg"))
    if RDKIT_AVAILABLE:
        print(f"dense RDKit features: {X_dense.shape}")
        return {"dense": X_dense}, {"dense": X_dense_test}

    X_sparse, X_sparse_test = sparse_text_pair(train_df, test_df)
    print(f"sparse features: {X_sparse.shape}, dense features: {X_dense.shape}")
    return {"sparse": X_sparse, "dense": X_dense}, {"sparse": X_sparse_test, "dense": X_dense_test}

## Models

Regressor definitions, shift diagnostics, and clipping.


In [21]:
def build_models(target_type, n_train):
    if RDKIT_AVAILABLE:
        models = []
    else:
        models = [
            ("ridge_005", "sparse", Pipeline([("model", Ridge(alpha=0.05, solver="lsqr", random_state=SEED))])),
            ("ridge_010", "sparse", Pipeline([("model", Ridge(alpha=0.10, solver="lsqr", random_state=SEED))])),
            ("ridge_030", "sparse", Pipeline([("model", Ridge(alpha=0.30, solver="lsqr", random_state=SEED))])),
            ("ridge_070", "sparse", Pipeline([("model", Ridge(alpha=0.70, solver="lsqr", random_state=SEED))])),
            ("svd_svr", "sparse", Pipeline([
                ("svd", TruncatedSVD(n_components=min(320 if target_type == "egc" else 256, n_train - 1), random_state=SEED)),
                ("scale", StandardScaler()),
                ("model", SVR(C=16.0 if target_type == "egc" else 8.0, epsilon=0.02 if target_type == "egc" else 1.5, gamma="scale")),
            ])),
        ]

    models += [
        ("hist_gbdt", "dense", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingRegressor(
                learning_rate=0.035, max_iter=380, l2_regularization=0.03,
                max_leaf_nodes=31, min_samples_leaf=8, random_state=SEED,
            )),
        ])),
    ]

    try:
        from xgboost import XGBRegressor
        models.append(("xgb_dense_3", "dense", XGBRegressor(
            n_estimators=550, learning_rate=0.025, max_depth=5,
            subsample=0.82, colsample_bytree=0.65, reg_alpha=0.02, reg_lambda=2.0,
            min_child_weight=1.0, objective="reg:squarederror", random_state=SEED + 22, n_jobs=-1,
        )))
    except Exception:
        pass

    try:
        from lightgbm import LGBMRegressor
        models.append(("lgb_dense", "dense", LGBMRegressor(
            n_estimators=720, learning_rate=0.018, num_leaves=31,
            min_child_samples=6, subsample=0.90, colsample_bytree=0.78,
            reg_alpha=0.02, reg_lambda=0.5, random_state=SEED, n_jobs=-1, verbose=-1,
        )))
        models.append(("lgb_dense_3", "dense", LGBMRegressor(
            n_estimators=680, learning_rate=0.022, num_leaves=15,
            min_child_samples=3, subsample=0.95, colsample_bytree=0.92,
            reg_alpha=0.05, reg_lambda=1.2, random_state=SEED + 32, n_jobs=-1, verbose=-1,
        )))
        models.append(("lgb_shift", "dense", LGBMRegressor(
            n_estimators=800, learning_rate=0.018, num_leaves=31,
            min_child_samples=7, subsample=0.92, colsample_bytree=0.82,
            reg_alpha=0.03, reg_lambda=0.7, random_state=SEED + 91,
            n_jobs=-1, verbose=-1,
        )))
    except Exception:
        pass

    return models

In [22]:
def adversarial_train_weights(X_train, X_test):
    """Estimate official-test density ratios without using test labels."""
    try:
        from lightgbm import LGBMClassifier
    except Exception:
        return np.ones(len(X_train), dtype=np.float32), 0.5
    X_domain = np.vstack([X_train, X_test]).astype(np.float32)
    domain = np.r_[np.zeros(len(X_train), dtype=np.int8), np.ones(len(X_test), dtype=np.int8)]
    oof = np.zeros(len(domain), dtype=np.float32)
    skf = StratifiedKFold(4, shuffle=True, random_state=SEED + 707)
    for tr_idx, va_idx in skf.split(X_domain, domain):
        clf = LGBMClassifier(
            n_estimators=350, learning_rate=0.035, num_leaves=23,
            min_child_samples=15, colsample_bytree=0.55,
            reg_lambda=1.5, random_state=SEED + 707, n_jobs=-1, verbose=-1,
        )
        clf.fit(X_domain[tr_idx], domain[tr_idx])
        oof[va_idx] = clf.predict_proba(X_domain[va_idx])[:, 1]
    auc = roc_auc_score(domain, oof)
    p = np.clip(oof[:len(X_train)], 0.03, 0.97)
    ratio = (p / (1.0 - p)) * (len(X_train) / len(X_test))
    weights = np.clip(ratio, 0.25, 4.0)
    weights /= weights.mean()
    print(f"adversarial AUC: {auc:.5f}; train-weight range: {weights.min():.2f}-{weights.max():.2f}")
    return weights.astype(np.float32), float(auc)

In [23]:
def clipped_predictions(preds, y):
    lo = np.nanpercentile(y, 0.25)
    hi = np.nanpercentile(y, 99.75)
    pad = 0.12 * (hi - lo)
    return np.clip(preds, lo - pad, hi + pad)

## Tg molecular kernels

Global binary and exact-count Tanimoto kernels.


In [24]:
def global_tanimoto_kernel_predictions(train_smiles, test_smiles, y, folds):
    """Winning Tg-only global Morgan/path Tanimoto SVR from OOF experiments."""
    all_smiles = list(train_smiles) + list(test_smiles)
    all_mols = [safe_mol(s) or Chem.MolFromSmiles("*") for s in all_smiles]
    n_train = len(train_smiles)

    def fp_matrix(kind, n_bits=2048):
        rows, cols = [], []
        for i, mol in enumerate(all_mols):
            if kind == "morgan2":
                fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=n_bits)
            elif kind == "morgan3":
                fp = AllChem.GetMorganFingerprintAsBitVect(mol, 3, nBits=n_bits)
            elif kind == "path":
                fp = Chem.RDKFingerprint(mol, fpSize=n_bits, maxPath=7)
            else:
                fp = rdMolDescriptors.GetHashedAtomPairFingerprintAsBitVect(mol, nBits=n_bits)
            bits = list(fp.GetOnBits())
            rows.extend([i] * len(bits)); cols.extend(bits)
        return sparse.csr_matrix(
            (np.ones(len(rows), dtype=np.float32), (rows, cols)),
            shape=(len(all_mols), n_bits),
        )

    def tanimoto(A, B):
        inter = (A @ B.T).toarray().astype(np.float32)
        an = np.asarray(A.sum(axis=1), dtype=np.float32).ravel()
        bn = np.asarray(B.sum(axis=1), dtype=np.float32).ravel()
        union = an[:, None] + bn[None, :] - inter
        return np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)

    def exact_log_count_matrix(radius=2):
        records=[]; vocabulary={}
        for mol in all_mols:
            record=AllChem.GetMorganFingerprint(mol,radius,useCounts=True).GetNonzeroElements()
            records.append(record)
            for key in record:
                if key not in vocabulary: vocabulary[key]=len(vocabulary)
        rows=[]; cols=[]; data=[]
        for i,record in enumerate(records):
            for key,count in record.items():
                rows.append(i); cols.append(vocabulary[key]); data.append(np.log1p(count))
        return sparse.csr_matrix((np.asarray(data,np.float32),(rows,cols)),
                                 shape=(len(all_mols),len(vocabulary)))

    def generalized_tanimoto(A, B):
        dot=(A@B.T).toarray().astype(np.float32)
        aa=np.asarray(A.multiply(A).sum(1),np.float32).ravel()
        bb=np.asarray(B.multiply(B).sum(1),np.float32).ravel()
        denom=aa[:,None]+bb[None,:]-dot
        return np.divide(dot,denom,out=np.zeros_like(dot),where=denom>1e-12)

    base = {}
    for kind in ["morgan2", "morgan3", "path", "atom_pair"]:
        F = fp_matrix(kind); A, B = F[:n_train], F[n_train:]
        base[kind] = (tanimoto(A, A), tanimoto(B, A))
    kernels = {
        "morgan2": base["morgan2"],
        "morgan_mix": (
            .55*base["morgan2"][0]+.45*base["morgan3"][0],
            .55*base["morgan2"][1]+.45*base["morgan3"][1]),
        "all_mix": (
            .45*base["morgan2"][0]+.25*base["morgan3"][0]+.30*base["path"][0],
            .45*base["morgan2"][1]+.25*base["morgan3"][1]+.30*base["path"][1]),
        "path_heavy": (
            .25*base["morgan2"][0]+.15*base["morgan3"][0]+.60*base["path"][0],
            .25*base["morgan2"][1]+.15*base["morgan3"][1]+.60*base["path"][1]),
        "atom_mix": (
            .35*base["morgan2"][0]+.20*base["path"][0]+.45*base["atom_pair"][0],
            .35*base["morgan2"][1]+.20*base["path"][1]+.45*base["atom_pair"][1]),
    }
    configs = [
        ("morgan2_C3", "morgan2", 3.0, .025),
        ("morgan_mix_C5", "morgan_mix", 5.0, .02),
        ("all_mix_C2", "all_mix", 2.0, .02),
        ("all_mix_C5", "all_mix", 5.0, .02),
        ("all_mix_C10", "all_mix", 10.0, .02),
        ("all_mix_eps05", "all_mix", 5.0, .05),
        ("path_heavy_C5", "path_heavy", 5.0, .02),
        ("atom_mix_C5", "atom_mix", 5.0, .02),
    ]
    mean, std = float(np.mean(y)), float(np.std(y)); z = (y - mean) / std
    results = []
    for name, key, c_value, epsilon in configs:
        train_kernel, test_kernel = kernels[key]
        oof = np.zeros(len(y), dtype=np.float32); test_folds = []
        for tr_idx, va_idx in folds:
            model = SVR(kernel="precomputed", C=c_value, epsilon=epsilon, cache_size=6000, tol=1e-3)
            model.fit(train_kernel[np.ix_(tr_idx, tr_idx)], z[tr_idx])
            oof[va_idx] = model.predict(train_kernel[np.ix_(va_idx, tr_idx)]) * std + mean
            test_folds.append(model.predict(test_kernel[:, tr_idx]) * std + mean)
        full_model=SVR(kernel="precomputed",C=c_value,epsilon=epsilon,cache_size=6000,tol=1e-3)
        full_model.fit(train_kernel,z)
        full_test=full_model.predict(test_kernel)*std+mean
        test_pred=.85*full_test+.15*np.mean(test_folds,axis=0)
        results.append((r2_score(y, oof), oof, test_pred, name))
    results.sort(key=lambda item: item[0], reverse=True)
    best = results[0][0]
    kept = [item for item in results if item[0] >= best - .01]
    weights = np.array([item[0] - (best - .01) + .001 for item in kept], dtype=np.float32)
    weights /= weights.sum()
    blend_oof = sum(w*item[1] for w, item in zip(weights, kept))
    blend_test = sum(w*item[2] for w, item in zip(weights, kept))
    print("Tg kernel components:", {item[3]: round(item[0], 5) for item in results})
    print(f"Tg multi-kernel blend: {r2_score(y, blend_oof):.5f}")

    count_features=exact_log_count_matrix(2)
    count_train,count_test=count_features[:n_train],count_features[n_train:]
    count_kernel=generalized_tanimoto(count_train,count_train)
    count_test_kernel=generalized_tanimoto(count_test,count_train)
    count_oof=np.zeros(len(y),dtype=np.float32); count_tests=[]
    for tr_idx,va_idx in folds:
        model=SVR(kernel="precomputed",C=5.0,epsilon=.02,cache_size=6000,tol=1e-3)
        model.fit(count_kernel[np.ix_(tr_idx,tr_idx)],z[tr_idx])
        count_oof[va_idx]=model.predict(count_kernel[np.ix_(va_idx,tr_idx)])*std+mean
        count_tests.append(model.predict(count_test_kernel[:,tr_idx])*std+mean)
    count_full=SVR(kernel="precomputed",C=5.0,epsilon=.02,cache_size=6000,tol=1e-3)
    count_full.fit(count_kernel,z)
    count_test_pred=.85*(count_full.predict(count_test_kernel)*std+mean)+.15*np.mean(count_tests,axis=0)
    print(f"Tg exact-count kernel: {r2_score(y,count_oof):.5f}")
    return blend_oof, blend_test, count_oof, count_test_pred

## Similarity helper

Optional molecular-neighbour implementation retained for reproducibility.


In [25]:
def tanimoto_neighbor_predictions(train_smiles, test_smiles, y, folds):
    """OOF and full-train predictions from the closest molecular analogues."""
    if not RDKIT_AVAILABLE:
        return []

    def fingerprints(smiles_values, kind):
        result = []
        for smiles in smiles_values:
            mol = safe_mol(str(smiles).replace("*", ""))
            if mol is None:
                result.append(None)
            elif kind == "morgan2":
                result.append(AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048))
            else:
                result.append(Chem.RDKFingerprint(mol, fpSize=2048, maxPath=7))
        return result

    def predict(query_fps, reference_fps, reference_y, k, power):
        out = np.empty(len(query_fps), dtype=np.float32)
        fallback = float(np.mean(reference_y))
        valid = [(fp, value) for fp, value in zip(reference_fps, reference_y) if fp is not None]
        if not valid:
            out.fill(fallback)
            return out
        valid_fps = [item[0] for item in valid]
        valid_y = np.asarray([item[1] for item in valid], dtype=np.float32)
        use_k = min(k, len(valid_fps))
        for i, fp in enumerate(query_fps):
            if fp is None:
                out[i] = fallback
                continue
            sims = np.asarray(DataStructs.BulkTanimotoSimilarity(fp, valid_fps), dtype=np.float32)
            top = np.argpartition(sims, -use_k)[-use_k:]
            weights = np.maximum(sims[top], 1e-4) ** power
            # Similarity-dependent shrinkage prevents remote neighbours from extrapolating.
            local = float(np.dot(weights, valid_y[top]) / weights.sum())
            confidence = float(np.clip((sims[top].max() - 0.20) / 0.55, 0.0, 1.0))
            out[i] = confidence * local + (1.0 - confidence) * fallback
        return out

    configurations = [
        ("knn_morgan_5", "morgan2", 5, 4.0),
        ("knn_morgan_15", "morgan2", 15, 3.0),
        ("knn_path_10", "path", 10, 3.0),
    ]
    cache = {}
    results = []
    for name, kind, k, power in configurations:
        if kind not in cache:
            cache[kind] = (
                fingerprints(train_smiles, kind), fingerprints(test_smiles, kind)
            )
        train_fps, test_fps = cache[kind]
        oof = np.zeros(len(y), dtype=np.float32)
        for tr_idx, va_idx in folds:
            ref_fps = [train_fps[i] for i in tr_idx]
            query_fps = [train_fps[i] for i in va_idx]
            oof[va_idx] = predict(query_fps, ref_fps, y[tr_idx], k, power)
        test_pred = predict(test_fps, train_fps, y, k, power)
        results.append((name, oof, test_pred))
    return results

## Training and submission

Cross-validation, full-data bagging, calibration, and submission generation.


In [26]:
def fit_target(train_part, test_part, target_type):
    print(f"\n=== {target_type.upper()} ===")
    Xs, Xs_test = build_feature_bundle(train_part, test_part, target_type)
    y = train_part["target"].to_numpy(dtype=np.float32)
    models = build_models(target_type, len(train_part))
    shift_weights, shift_auc = adversarial_train_weights(Xs["dense"], Xs_test["dense"])
    kf = KFold(n_splits=min(N_SPLITS, len(train_part)), shuffle=True, random_state=SEED)
    folds = list(kf.split(np.arange(len(train_part))))

    names, oofs, test_preds, scores = [], [], [], []
    for name, kind, estimator in models:
        print(f"training {name}...")
        X = Xs[kind]
        X_test = Xs_test[kind]
        oof = np.zeros(len(train_part), dtype=np.float32)
        fold_test = []
        for fold, (tr_idx, va_idx) in enumerate(folds, 1):
            model = clone(estimator)
            if name == "lgb_shift":
                model.fit(X[tr_idx], y[tr_idx], sample_weight=shift_weights[tr_idx])
            else:
                model.fit(X[tr_idx], y[tr_idx])
            oof[va_idx] = clipped_predictions(model.predict(X[va_idx]), y)
            fold_test.append(model.predict(X_test))
        score = r2_score(y, oof)
        print(f"{name:>14}: CV R2 = {score:.5f}")
        full_predictions=[]
        for bag_seed in [SEED + 1001, SEED + 2003, SEED + 3007]:
            full_model=clone(estimator)
            seed_params={key:bag_seed for key in full_model.get_params() if key.endswith("random_state")}
            if seed_params: full_model.set_params(**seed_params)
            if name == "lgb_shift":
                full_model.fit(X, y, sample_weight=shift_weights)
            else:
                full_model.fit(X, y)
            full_predictions.append(full_model.predict(X_test))
        full_test_pred = np.mean(full_predictions, axis=0)
        fold_test_pred = np.mean(fold_test, axis=0)
        names.append(name)
        oofs.append(oof)
        test_preds.append(0.85 * full_test_pred + 0.15 * fold_test_pred)
        scores.append(score)

    if target_type == "tg":
        print("training Tg global Tanimoto kernel...")
        kernel_oof, kernel_test, count_oof, count_test = global_tanimoto_kernel_predictions(
            train_part["smiles"].tolist(), test_part["smiles"].tolist(), y, folds
        )
        kernel_score = r2_score(y, kernel_oof)
        print(f"{'tanimoto_svr':>14}: CV R2 = {kernel_score:.5f}")
        names.append("tanimoto_svr"); oofs.append(kernel_oof)
        test_preds.append(kernel_test); scores.append(kernel_score)
        count_score = r2_score(y, count_oof)
        print(f"{'count_kernel':>14}: CV R2 = {count_score:.5f}")
        names.append("count_kernel"); oofs.append(count_oof)
        test_preds.append(count_test); scores.append(count_score)


    score_values = np.array(scores, dtype=np.float32)
    best = float(score_values.max())
    keep_margin = 0.025 if RDKIT_AVAILABLE else 0.04
    keep = score_values >= best - keep_margin
    if keep.sum() < 2:
        keep[np.argsort(score_values)[-min(2, len(score_values)):]] = True
    shifted = score_values - score_values[keep].min() + 0.001
    weights = np.where(keep, shifted, 0.0)
    weights = weights / weights.sum()

    oof_matrix = np.vstack(oofs).T
    test_matrix = np.vstack(test_preds).T
    weighted_oof = np.average(oof_matrix, axis=1, weights=weights)
    weighted_test = np.average(test_matrix, axis=1, weights=weights)
    weighted_score = r2_score(y, weighted_oof)
    print(f"{'weighted_blend':>14}: CV R2 = {weighted_score:.5f}")
    print("Blend weights:", dict(zip(names, np.round(weights, 4))))

    selected_pred = weighted_test
    selected_oof = weighted_oof
    if keep.sum() >= 2:
        kept = np.where(keep)[0]
        stacker = LinearRegression(positive=True)
        stacker.fit(oof_matrix[:, kept], y)
        stack_oof = clipped_predictions(stacker.predict(oof_matrix[:, kept]), y)
        stack_score = r2_score(y, stack_oof)
        print(f"{'positive_stack':>14}: CV R2 = {stack_score:.5f}")
        if stack_score > weighted_score:
            stack_test = test_matrix[:, kept].dot(stacker.coef_) + float(stacker.intercept_)
            # Shrink the fitted stack toward a fixed blend; this transfers better under split shift.
            selected_pred = 0.65 * stack_test + 0.35 * weighted_test
            robust_oof = 0.65 * stack_oof + 0.35 * weighted_oof
            selected_oof = robust_oof
            print(f"{'robust_blend':>14}: CV R2 = {r2_score(y, robust_oof):.5f}")
            print(f"{'selected':>14}: robust_blend")
        else:
            print(f"{'selected':>14}: weighted_blend")
    else:
        print(f"{'selected':>14}: weighted_blend")

    # When train/test structures are distinguishable, reserve a modest contribution
    # for the density-ratio model even if ordinary random CV undervalues it.
    if "lgb_shift" in names and shift_auc >= 0.58:
        shift_idx = names.index("lgb_shift")
        alpha = 0.20 if shift_auc < 0.68 else 0.30
        selected_pred = (1.0 - alpha) * selected_pred + alpha * test_matrix[:, shift_idx]
        shifted_oof = (1.0 - alpha) * weighted_oof + alpha * oof_matrix[:, shift_idx]
        print(f"shift-aware contribution: {alpha:.2f}; diagnostic random-CV R2: {r2_score(y, shifted_oof):.5f}")
    else:
        print("shift-aware contribution: 0.00")
    calibrator = LinearRegression().fit(selected_oof.reshape(-1, 1), y)
    calibrated_oof = calibrator.predict(selected_oof.reshape(-1, 1))
    calibrated_score = r2_score(y, calibrated_oof)
    base_selected_score = r2_score(y, selected_oof)
    if calibrated_score > base_selected_score + 1e-5:
        selected_pred = calibrator.predict(np.asarray(selected_pred).reshape(-1, 1))
        print(f"OOF calibration: slope={calibrator.coef_[0]:.5f}, intercept={calibrator.intercept_:.5f}, R2={calibrated_score:.5f}")
    else:
        print("OOF calibration: not applied")
    return clipped_predictions(selected_pred, y)

In [27]:
train = pd.read_csv(find_input_file("train.csv"))
test = pd.read_csv(find_input_file("test.csv"))
train["target_type"] = train["target_type"].str.lower()
test["target_type"] = test["target_type"].str.lower()

submission = test[["id"]].copy()
submission["target"] = np.nan

for target_type in sorted(train["target_type"].unique()):
    train_part = train.loc[train["target_type"] == target_type].reset_index(drop=True)
    test_mask = test["target_type"] == target_type
    test_part = test.loc[test_mask].reset_index(drop=True)
    if len(test_part):
        submission.loc[test_mask, "target"] = fit_target(train_part, test_part, target_type)

if submission["target"].isna().any():
    fallback = train.groupby("target_type")["target"].median().to_dict()
    for idx in submission.index[submission["target"].isna()]:
        submission.loc[idx, "target"] = fallback.get(test.loc[idx, "target_type"], train["target"].median())

submission = submission.sort_values("id").reset_index(drop=True)
submission.to_csv("submission.csv", index=False)
print("\nSaved submission.csv")
print(submission.head())


=== EGC ===
dense RDKit features: (2028, 8182)
adversarial AUC: 0.50918; train-weight range: 0.21-3.31
training hist_gbdt...
     hist_gbdt: CV R2 = 0.90301
training xgb_dense_3...
   xgb_dense_3: CV R2 = 0.90886
training lgb_dense...
     lgb_dense: CV R2 = 0.90630
training lgb_dense_3...
   lgb_dense_3: CV R2 = 0.90263
training lgb_shift...
     lgb_shift: CV R2 = 0.90400
weighted_blend: CV R2 = 0.90970
Blend weights: {'hist_gbdt': np.float32(0.0828), 'xgb_dense_3': np.float32(0.4342), 'lgb_dense': np.float32(0.2804), 'lgb_dense_3': np.float32(0.06), 'lgb_shift': np.float32(0.1426)}
positive_stack: CV R2 = 0.91088
  robust_blend: CV R2 = 0.91069
      selected: robust_blend
shift-aware contribution: 0.00
OOF calibration: slope=1.01004, intercept=-0.04812, R2=0.91079

=== TG ===
dense RDKit features: (4143, 10230)
adversarial AUC: 0.49884; train-weight range: 0.22-3.58
training hist_gbdt...
     hist_gbdt: CV R2 = 0.90228
training xgb_dense_3...
   xgb_dense_3: CV R2 = 0.89752
traini